# **Introduction**

In this assignment, we will build a **complete machine learning pipeline** for **news topic classification**. Each English news text will be classified into one of four categories: **World, Sports, Business, or Sci/Tech**.

**Project topic:** News Topic Classification: A Comparative Study of Traditional Machine Learning and Deep Learning Approaches.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fufu3105/machine-learning-assignment/blob/main/assignment.ipynb)

**Source code:** [GitHub repository](https://github.com/fufu3105/machine-learning-assignment).

The pipeline consists of the following main stages:

1. **EDA (Exploratory Data Analysis):** Explore the dataset structure, class distribution, text-length statistics, word frequencies, duplicate samples, and potential outliers.
2. **Data Preprocessing:** Check missing and duplicate texts, configure text normalization and stopword removal for each representation, and create a validation split from the predefined training set while keeping the test set separate.
3. **Feature Extraction and Representation:** Apply traditional text representations such as **Bag-of-Words (BoW), TF-IDF, and n-grams**, together with modern pretrained representations such as **GloVe and DistilBERT embeddings**.
4. **Model Training:** Train and compare multiple traditional machine learning classifiers, including **Logistic Regression, Support Vector Machine (SVM), Naive Bayes, and Random Forest**.
5. **Model Evaluation:** Compute **Accuracy, Macro Precision, Macro Recall, Macro F1, and Weighted F1**, display **Confusion Matrices**, and compare training and inference time where appropriate. Use the validation set to choose configurations and reserve the test set for final evaluation.
6. **End-to-End Deep Learning:** Develop deep learning models such as **BiLSTM** and **fine-tuned DistilBERT** and compare their performance with traditional machine learning approaches.

## **Dataset**

* **Dataset:** [AG News](https://huggingface.co/datasets/fancyzhx/ag_news)
* **Domain:** English-language news articles from multiple news categories.
* **Task:** Multi-class text classification.
* **Number of classes:** 4.
* **Classes:** `World`, `Sports`, `Business`, and `Sci/Tech`.
* **Input:** The `text` field provided by the Hugging Face version of AG News.
* **Target:** News topic `label`.
* **Data split:** 120,000 training samples and 7,600 testing samples.

AG News provides a sufficiently large and diverse benchmark dataset for systematically evaluating both traditional text classification techniques and modern deep learning approaches.

## **Objectives**

* Build a configurable **traditional machine learning pipeline** for multi-class text classification.
* Compare multiple traditional feature extraction techniques, including **BoW, TF-IDF, and n-grams**.
* Investigate modern text representations using pretrained **GloVe** and **DistilBERT embeddings**.
* Save extracted feature representations as **`.npy` or `.h5` files** for downstream classification and submission.
* Train and compare different traditional classifiers, including **Logistic Regression, SVM, Naive Bayes,** and **Random Forest**.
* Document the tested preprocessing, representation, and model parameter configurations together with their measured performance.
* Implement **end-to-end deep learning models** using **BiLSTM** and **fine-tuned DistilBERT** as an extension to the traditional pipeline.
* Compare traditional and deep learning approaches using both predictive performance and computational considerations.
* Analyze the advantages and limitations of different text representations and modeling approaches for news topic classification.

The overall experimental pipeline is designed to answer the following question:

> **How do traditional sparse text representations, pretrained semantic embeddings and end-to-end deep learning approaches compare in news topic classification?**


# Library Installation and Imports

In this step, we will:

- Install the libraries needed for the traditional machine learning and deep learning pipelines.
- Import tools for EDA, text processing, feature extraction, model training, and evaluation.

**Google Colab:** Open the notebook using the badge above and select **Runtime > Run all**. Use the hosted Python 3 runtime; a CPU is sufficient for this section. Libraries are installed in the first code cell, and the required directories are created automatically.


In [1]:
# Install the required libraries in the active local or Google Colab kernel.
%pip install -q numpy pandas matplotlib seaborn wordcloud nltk scikit-learn datasets transformers "gensim>=4.4.0" torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ------------------------------------------------------------
# Import utilities for paths, random seeds, and timing.
# ------------------------------------------------------------
import random
from pathlib import Path
from time import perf_counter


# ------------------------------------------------------------
# Import tools for numerical analysis, tables, and text visualization.
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# ------------------------------------------------------------
# Prepare text processing tools and interfaces for later data and GloVe loading.
# ------------------------------------------------------------
import nltk
from nltk.corpus import stopwords
from datasets import load_dataset
import gensim.downloader as gensim_api

# ------------------------------------------------------------
# Import traditional text representations, classifiers, and evaluation tools.
# ------------------------------------------------------------
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# ------------------------------------------------------------
# Prepare PyTorch and Hugging Face tools for later deep learning experiments.
# ------------------------------------------------------------
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
)


In [3]:
# Set the random seed for reproducibility.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Use a GPU when available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create directories for features, models, and results.
FEATURE_DIR = Path("features")
MODEL_DIR = Path("models")
RESULT_DIR = Path("results")

for directory in (FEATURE_DIR, MODEL_DIR, RESULT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Seed: {SEED}")
print(f"Device: {device}")


Seed: 42
Device: cpu


# Dataset Loading

In this step, we will load [AG News](https://huggingface.co/datasets/fancyzhx/ag_news) from Hugging Face and store the dataset cache in **`data/raw/ag_news/`**, preserving the official **train** and **test** splits.


In [5]:
DATASET_ID = "fancyzhx/ag_news"
DATA_DIR = Path("data") / "raw" / "ag_news"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Download and cache AG News in the local data directory.
dataset = load_dataset(DATASET_ID, cache_dir=str(DATA_DIR))


NameError: name 'fancyzhx' is not defined